# Model Eval; Grade Visualisation

## How to run the grading

Grading is performed by Claude Code using `data/results/grading_spec.md` as the scoring guide.

**Prerequisite:** `data/results/model_eval.json` must exist; produced by running `04_model_eval.ipynb`.

**Run the grading** from the project root:

```
claude
```

Then paste this prompt:

```
Read `data/results/grading_spec.md` and `data/results/model_eval.json`, then grade every
record according to the criteria below. Write the results to `data/results/model_eval_grades.json`.
```

Claude reads both files, scores each record on the task-specific criteria (1–5), and writes
`data/results/model_eval_grades.json`; metadata + scores only, no raw results.

Once that file is present, run the cells below.

In [3]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

with open('../data/results/model_eval_grades.json') as f:
    records = json.load(f)

rows = []
for r in records:
    print(r)
    base = {
        'model': r['model'],
        'task': r['task'],
        'doc': r['doc'],
        'elapsed_s': r['elapsed_s'],
    }
    if r['scores']:
        for k, v in r['scores'].items():
            if k != 'notes':
                rows.append({**base, 'criterion': k, 'score': v})

df = pd.DataFrame(rows)
print(df.shape, df['model'].nunique(), 'models')

{'model': 'qwen3:8b', 'task': 'ner', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 60.3, 'scores': None}
{'model': 'qwen3:8b', 'task': 'tags', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 61.3, 'scores': None}
{'model': 'qwen3:8b', 'task': 'summary', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 61.9, 'scores': None}
{'model': 'qwen3:14b', 'task': 'ner', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 70.8, 'scores': {'entity_form': 4, 'type_accuracy': 4, 'hallucination': 4, 'notes': 'Clean forms, correct types; sparse (8 entities); Emeti Yakuf not verified in source portion read'}}
{'model': 'qwen3:14b', 'task': 'tags', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 37.1, 'scores': None}
{'model': 'qwen3:14b', 'task': 'summary', 'doc': '20241217_192002_Turkistan Islamic Party - Wikiwand.md', 'elapsed_s': 23.7, 'scores': None}
{'model'

In [4]:
# Heatmap: average score per model × criterion
pivot = df.groupby(['model', 'criterion'])['score'].mean().unstack()

# Order criteria by task group for readability
criterion_order = [
    'entity_form', 'type_accuracy', 'hallucination',   # NER
    'relevance', 'style', 'language', 'coverage',      # tags (coverage shared with summary)
    'factual_accuracy', 'specificity',                  # summary
]
criterion_order = [c for c in criterion_order if c in pivot.columns]
pivot = pivot[criterion_order]

# Order models alphabetically
model_order = sorted(pivot.index.tolist())
pivot = pivot.loc[model_order]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=1, zmax=5,
    text=pivot.round(1).values,
    texttemplate='%{text}',
    hovertemplate='model: %{y}criterion: %{x}avg score: %{z:.2f}',
))
fig.update_layout(
    title='Average score per model × criterion (across both docs)',
    xaxis_title='Criterion',
    yaxis_title='Model',
    height=800,
    margin=dict(l=140),
)
fig.show()

In [7]:
# Grouped bar chart: mean score per model, faceted by task
task_mean = (
    df.groupby(['model', 'task'])['score']
    .mean()
    .reset_index()
    .rename(columns={'score': 'mean_score'})
)

# Order models alphabetically
model_order = sorted(df['model'].unique().tolist())

fig = px.bar(
    task_mean,
    x='model',
    y='mean_score',
    facet_col='task',
    category_orders={'model': model_order, 'task': ['ner', 'tags', 'summary']},
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    labels={'mean_score': 'Mean score', 'model': 'Model'},
    title='Mean score per task and model (averaged across both docs and task criteria)',
    height=400,
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
fig.update_yaxes(range=[0, 5.2])
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].upper()))
fig.show()

In [6]:
# Scatter: mean elapsed_s vs mean quality score (speed / quality tradeoff)
agg = (
    df.groupby('model')
    .agg(mean_score=('score', 'mean'), mean_elapsed=('elapsed_s', 'mean'))
    .reset_index()
)

fig = px.scatter(
    agg,
    x='mean_elapsed',
    y='mean_score',
    text='model',
    labels={'mean_elapsed': 'Mean elapsed (s)', 'mean_score': 'Mean score'},
    title='Speed vs quality tradeoff per model',
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    height=450,
)
fig.update_traces(textposition='top center', marker_size=10)
fig.update_layout(yaxis_range=[1, 5.2], coloraxis_showscale=False)
fig.show()

## Conclusion

Grades cover 27 models across 3 tasks (NER, tags, summary) and 7 documents (EN + NL), yielding 567 records of which 472 were scored and 95 failed to produce valid output. Individual criterion scores total 1,744.

### Best overall (reliable): `gemma3:12b` — mean 4.41 / 5

Best performer among models with near-complete coverage (1 null record out of 21). NER 4.22, tags 4.38, summary 4.50. Hallucination score of 4.83 on NER is second only to `phi4-reasoning:14b` and `dolphin3:8b`. Mean latency 14.4 s — fastest in the top tier.

### Highest raw mean: `qwen3:14b` — mean 4.63 / 5 (unreliable)

Scores best on completed records — tags 4.75, summary 4.79 — but failed 5 of 21 records, all on NER tasks across multiple documents. Mean latency 50.5 s. If NER is not required, it is the strongest option.

### Best per task

- **NER**: `dolphin3:8b` — 4.38, 0 failures, 10.4 s. Cleanest entity forms and zero hallucinations.
- **Tags**: `qwen3:14b` — 4.75 (5 failures) / `deepseek-r1:32b` — 4.46 reliable but 349 s mean latency.
- **Summary**: `deepseek-r1:32b` — 4.68 (1 failure, 349 s) / `phi4:14b` — 4.54, 2 failures, 23.3 s.

### Notable observations

- **Essentially failed**: `qwen3-vl:30b` (20/21 null) and `olmo-3:7b` (15/21 null) could not complete most tasks and should be excluded from production consideration.
- **High failure rate**: `deepseek-r1:8b` (10 failures), `falcon:7b` (9), `qwen3:8b` (7), `granite4:7b-a1b-h` (5) all produced null output on a significant fraction of records.
- **Hallucinations**: `falcon:7b` scored 1.0 on NER hallucination (invented entities on every record). `gemma2:2b` (2.0) and `lfm2:24b` (2.67) also fabricated entities regularly. All other models scored 3.0 or above.
- **NER entity form** remains the weakest criterion across the board: most models append honorifics, job titles, or context phrases to entity names, pulling average NER scores below tags and summary.
- **Speed vs quality**: `gemma3:12b` offers the best balance — top-tier quality at 14 s. `deepseek-r1:32b` matches it on summary but is 24x slower at 349 s mean latency.

## Failure analysis

95 of 567 records (17%) produced no valid output. Three distinct failure modes account for almost all of them.

### 1. Empty response (50 failures, 53%)

The model returned nothing parseable. Two sub-causes:

- **Content refusal**: The Turkistan Islamic Party doc triggers safety filters in several models (`qwen3:8b`, `qwen3:14b`, `olmo-3:7b`, `deepseek-r1:8b`). The model spends the full inference budget, then outputs a refusal message or nothing instead of JSON. That document has a 36% failure rate vs 14% across all other docs.
- **Blank output**: Some models emit an empty string on certain inputs without any apparent safety reason.

### 2. Truncated JSON (20 failures, 21%)

`Unterminated string` errors indicate the model started producing valid JSON but stopped mid-stream before closing the structure. Likely cause: hitting a context window or token-generation limit. Affected models: `solar:10.7b`, `deepseek-r1:8b`, `falcon:7b`, `gemma3n:e4b`.

### 3. Wrong output structure (19 failures, 20%)

`KeyError: 'tags'` / `'summary'` / `'entities'` - the model returned valid JSON but used different key names than expected (e.g. `result`, `output`, `named_entities`). The eval code fails hard on key lookup instead of falling back. Most common in `llama3.1:8b`, `llama3.2:latest`, `granite3.3:8b`, `granite4:7b-a1b-h`, `lfm2:24b`.

### 4. Malformed JSON (6 failures)

Invalid comma placement or bad Unicode escape sequences. Models that don't reliably produce syntactically valid JSON: `falcon:7b`, `llama3.2:latest`.